In [1]:
import duckdb, pandas as pd, os, csv, re

TABELAS = {
    'controle_pessoa': 'dados/controle de pessoa.csv',
    'atendimentos': [
        'dados/tbl_101.csv', 'dados/tbl_1144.csv',
        'dados/tbl_1763.csv', 'dados/tbl_1976.csv', 'dados/tbl_2010.csv',
    ],
    'workflow': 'dados/WF.csv',
}
BANCO = 'meu_banco.duckdb'; SEP = ';'; ENC = 'latin1'


def ler_csv(caminho):
    tam = os.path.getsize(caminho) / 1_048_576
    print(f'    {os.path.basename(caminho)} ({tam:.1f} MB)...')
    linhas = []; corrigidos = 0
    with open(caminho, 'r', encoding=ENC, errors='replace', newline='') as f:
        rd = csv.reader(f, delimiter=SEP)
        header = next(rd)
        n = len(header)
        for row in rd:
            if len(row) == n + 1:
                # Coluna extra no inicio (formula =T(...)): extrair numero e realinhar
                num = re.search(r'\d{10,}', str(row[0]))
                row = [num.group(0) if num else ''] + row[1:]
                corrigidos += 1
            if len(row) > n:
                row = row[:n]
            elif len(row) < n:
                row += [''] * (n - len(row))
            linhas.append(row)
    if corrigidos:
        print(f'    [LIMPEZA] {corrigidos:,} linhas com coluna =T(...) realinhadas')
    return pd.DataFrame(linhas, columns=header)


def importar_tabela(con, nome, fontes):
    if isinstance(fontes, str): fontes = [fontes]
    partes = [ler_csv(a) for a in fontes if os.path.exists(a)]
    if not partes:
        print(f'  [ERRO] Nenhum arquivo para "{nome}".'); return 0
    df = pd.concat(partes, ignore_index=True) if len(partes) > 1 else partes[0]
    con.register('_tmp_', df)
    con.execute(f'CREATE OR REPLACE TABLE "{nome}" AS SELECT * FROM _tmp_')
    con.unregister('_tmp_')
    total = con.execute(f'SELECT COUNT(*) FROM "{nome}"').fetchone()[0]
    print(f'  [OK] "{nome}" -> {total:,} registros | {len(df.columns)} colunas')
    return total


con = duckdb.connect(BANCO)
print(f'Banco: {BANCO}  |  Importando {len(TABELAS)} tabela(s)...\n')
resumo = {}
for nome, arqs in TABELAS.items():
    qtd = 1 if isinstance(arqs, str) else len(arqs)
    print(f'>> {nome} ({qtd} arquivo(s))')
    resumo[nome] = importar_tabela(con, nome, arqs)
    print()
print(f'Total: {sum(resumo.values()):,} registros em {len(resumo)} tabelas')
display(con.execute('SHOW TABLES').df())


Banco: meu_banco.duckdb  |  Importando 3 tabela(s)...

>> controle_pessoa (1 arquivo(s))
    controle de pessoa.csv (0.2 MB)...
  [OK] "controle_pessoa" -> 396 registros | 63 colunas

>> atendimentos (5 arquivo(s))
    tbl_101.csv (77.6 MB)...
    tbl_1144.csv (6.0 MB)...
    tbl_1763.csv (1.8 MB)...
    tbl_1976.csv (0.8 MB)...
    tbl_2010.csv (256.3 MB)...
  [OK] "atendimentos" -> 1,351,505 registros | 42 colunas

>> workflow (1 arquivo(s))
    WF.csv (257.4 MB)...
    [LIMPEZA] 196,984 linhas com coluna =T(...) realinhadas
  [OK] "workflow" -> 196,984 registros | 41 colunas

Total: 1,548,885 registros em 3 tabelas


,name
0,atendimentos
1,controle_pessoa
2,workflow


In [2]:
for nome in TABELAS:
    total = con.execute(f'SELECT COUNT(*) FROM "{nome}"').fetchone()[0]
    print(f'--- {nome}: {total:,} registros ---')
    display(con.execute(f'SELECT * FROM "{nome}" LIMIT 3').df())
    print()


--- controle_pessoa: 396 registros ---


,MAT.,LOGIN,NOME,DATA DE NASCIMENTO,STATUS,TIPO,COORDENADOR/SUPERVISOR/GERENTE,COORDENADOR/GERENTE/DIRETOR,SETORES,CELULA,...,PREVISAO,RETORNO_2,INICIO_4,RETORNO_3,INICIO_5,RETORNO_4,EMPRESA,GERÊNCIAS,INÍCIO DE TREINAMENTO,UNIDADE DE ORIGEM
0,4871,SUZANE,SUZANE LIMA DA SILVA,23/04/1981,EMPRESTADO,EFETIVO,ADRIANA NOBRE,VANESSA ALENCAR,GECRE,GECRE,...,,,,,,,-,GERENCIA DE CREDENCIAMENTO,,-
1,4686,ALINESOU,ALINE DE SOUZA ALVES,06/11/1989,ATIVO,EFETIVO,ANDRESSA MEDEIROS,PRISCILA LAGE,MULTICANAIS,CANAIS VIRTUAIS,...,,,,,,,-,GERENCIA DE RELACIONAMENTO OPERADORA ASSIM SAUDE,,-
2,90531,EVALERIA,VALERIA PINTO ELEUTERIO,24/05/1978,FERIAS,EFETIVO,ANDRESSA MEDEIROS,PRISCILA LAGE,MULTICANAIS,CANAIS VIRTUAIS,...,,,,,,,-,GERENCIA DE RELACIONAMENTO OPERADORA ASSIM SAUDE,,-



--- atendimentos: 1,351,505 registros ---


,Tipo Atendimento,No.Atendimento,Operador,Data de entrada,Matrícula,Nome,Data da proposta,Email,Telefone,Endereço,...,Código do Grupo,Grupo,Especialidade,Código do Executor,Executor,Código do Grupo_1,Grupo_1,CID,Diagnóstico,TUSS Inicial
0,258-TELE - EMAIL DO ATENDIMENTO,30922220260101001372,DENISER,01/01/2026,447076.0000002.0,KAROLINNE RANGEL RISCADO ARRUDA,07/06/2024,KAROLARRUDA50@GMAIL.COM,(22)0000-0000,RUA. JARDIM BOTANICO 145.,...,,,,,,,,,,
1,258-TELE - EMAIL DO ATENDIMENTO,30922220260101001414,DENISER,01/01/2026,000000.0029065.0,MARIA MOREIRA DE ABREU,25/06/1990,ANAPAULAMOREIRADEABREU@GMAIL.COM,3232182604,"RUA GOIAS,27",...,,,,,,,,,,
2,258-TELE - EMAIL DO ATENDIMENTO,30922220260101001465,DENISER,01/01/2026,004653.0046776.0,MARIA DE FATIMA VICTOR DA SILVA,01/06/2015,VICTORDASILVAMARIADEFATIMA@YAHOO.COM.BR,21 25806092,RUA COUTO DE MAGALHAES 125 CASA 11,...,,,,,,,,,,



--- workflow: 196,984 registros ---


,Protocolo,Protocolo_ID,Matricula,Nome do Beneficiario,Tipo de Atendimento,Tipo Macro,Tipo Ocorrencia,Motivo,Submotivo,Prioridade,...,Data Final Etapa,Hora Final Etapa,Status da Etapa,Prazo Planejado Etapa,Prazo Real Etapa,SLA Etapa,Ocorrencia,Providencia,Bairro,Cidade
0,30922220251113000207,30922220251113000207_ 1,4653.16288.0,LINAURIA GOMES DE ANDRADE,1544-TRANS_HOSPITAL_PUBLICE_PARTICULAR*,SOLICITAÇÃO,ATENDIMENTO,TRANSFERENCIA,HOSPITAL PUBLICO OU PARTICULAR,,...,13/11/2025,00:17:15,PENDENTE,0,0,DENTRO DO PRAZO,NOME DO BENEFICIÁRIO:LINAURIA GOMES DE ANDRADE...,,RAMOS,RIO DE JANEIRO
1,30922220251113000207,30922220251113000207_ 2,4653.16288.0,LINAURIA GOMES DE ANDRADE,1544-TRANS_HOSPITAL_PUBLICE_PARTICULAR*,SOLICITAÇÃO,ATENDIMENTO,TRANSFERENCIA,HOSPITAL PUBLICO OU PARTICULAR,,...,13/11/2025,00:44:43,ATENDIDO,0,0,DENTRO DO PRAZO,NOME DO BENEFICIÁRIO:LINAURIA GOMES DE ANDRADE...,"prezados, boa noite! daremos andamento confor...",RAMOS,RIO DE JANEIRO
2,30922220251113001172,30922220251113001172_ 1,472133.1.0,NATALIA BARROSO PINHEIRO DE ANDRADE,269-AMEACA*,RECLAMAÇÃO,ATENDIMENTO,AMEAÇA INICIAL,AMEAÇA INICIAL,,...,13/11/2025,03:40:15,PENDENTE,0,0,DENTRO DO PRAZO,NOME DO CLIENTE: NATALIA BARROSO PINHEIRO DE A...,,CAMPO GRANDE,RIO DE JANEIRO


In [3]:
# =============================================================
# PREPARACAO DOS DADOS PARA ANALISE DE PRODUTIVIDADE
# Fonte: meu_banco.duckdb (workflow + atendimentos + controle_pessoa)
# =============================================================
import pandas as pd

# Carrega as tabelas do DuckDB (con ja esta aberto da Celula 1)
c_pessoal = con.execute('SELECT * FROM controle_pessoa').df()
relatorio  = con.execute('SELECT * FROM workflow').df()
saps       = con.execute('SELECT * FROM atendimentos').df()

# Garante colunas limpas
c_pessoal.columns = c_pessoal.columns.str.strip()
relatorio.columns = relatorio.columns.str.strip()
saps.columns      = saps.columns.str.strip()

print(f'Workflow  : {len(relatorio):>10,} registros')
print(f'SAPS      : {len(saps):>10,} registros')
print(f'Controle  : {len(c_pessoal):>10,} registros')


Workflow  :    196,984 registros
SAPS      :  1,351,505 registros
Controle  :        396 registros


In [4]:
# Coluna MES no Workflow
# Apos o realinhamento do =T(...), 'Data de Entrada' contem a data real (ex: 13/11/2025)
relatorio['Data de Entrada'] = pd.to_datetime(relatorio['Data de Entrada'], dayfirst=True, errors='coerce')
relatorio['MES'] = relatorio['Data de Entrada'].dt.strftime('%y-%m')

# Verifica o range de datas do workflow
datas_wf = relatorio['Data de Entrada'].dropna()
print(f'Workflow  | Data de Entrada: {datas_wf.min().strftime("%d/%m/%Y")} -> {datas_wf.max().strftime("%d/%m/%Y")}')
print(f'          | Meses distintos: {relatorio["MES"].nunique()}')

# Coluna MES no SAPS
coluna_data_saps = 'Data de entrada'
saps[coluna_data_saps] = pd.to_datetime(saps[coluna_data_saps], dayfirst=True, errors='coerce')
saps['MES'] = saps[coluna_data_saps].dt.strftime('%y-%m')

datas_saps = saps[coluna_data_saps].dropna()
print(f'SAPS      | Data de entrada: {datas_saps.min().strftime("%d/%m/%Y")} -> {datas_saps.max().strftime("%d/%m/%Y")}')
print(f'          | Meses distintos: {saps["MES"].nunique()}')

# Lista de logins ativos
logins = c_pessoal['LOGIN'].dropna().str.strip().tolist()
info_pessoal = (
    c_pessoal[['LOGIN', 'STATUS', 'TIPO', 'COORDENADOR/SUPERVISOR/GERENTE']]
    .rename(columns={
        'STATUS': 'Status',
        'TIPO': 'Tipo',
        'COORDENADOR/SUPERVISOR/GERENTE': 'Supervisor'
    })
)
print(f'Logins ativos no controle: {len(logins)}')


Workflow  | Data de Entrada: 13/11/2025 -> 11/05/2026
          | Meses distintos: 7
SAPS      | Data de entrada: 01/01/2026 -> 11/05/2026
          | Meses distintos: 5
Logins ativos no controle: 396


In [5]:
# ── WORKFLOW: produtividade por operador x mes ──────────────────
rel_filtrado = relatorio[relatorio['Usuario Mov.'].isin(logins)]

produtividade_filtrada = pd.crosstab(
    index=rel_filtrado['Usuario Mov.'],
    columns=rel_filtrado['MES'],
    margins=True, margins_name='TOTAL'
)
produtividade_filtrada = pd.concat([
    produtividade_filtrada.drop('TOTAL').sort_values(by='TOTAL', ascending=False),
    produtividade_filtrada.loc[['TOTAL']]
])
produtividade_filtrada = produtividade_filtrada.reset_index()
produtividade_filtrada = pd.merge(produtividade_filtrada, info_pessoal,
                                   left_on='Usuario Mov.', right_on='LOGIN', how='left')
produtividade_filtrada = produtividade_filtrada.drop('LOGIN', axis=1)
colunas_mes = [c for c in produtividade_filtrada.columns
               if c not in ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor', 'TOTAL']]
produtividade_filtrada = produtividade_filtrada[['Usuario Mov.', 'Status', 'Tipo', 'Supervisor'] + colunas_mes + ['TOTAL']]

print(f'Workflow filtrado: {len(produtividade_filtrada):,} operadores x {len(colunas_mes)} meses')


Workflow filtrado: 351 operadores x 7 meses


In [6]:
# ── SAPS: produtividade por operador x mes ──────────────────────
# Empilha Operador e Operador Final (cada atendimento pode contar para 2 operadores)
saps_long = pd.concat([
    saps[['Operador', 'MES']].assign(idx=saps.index).rename(columns={'Operador': 'login'}),
    saps[['Operador Final', 'MES']].assign(idx=saps.index).rename(columns={'Operador Final': 'login'})
])
saps_long = saps_long.dropna(subset=['login'])
saps_long['login'] = saps_long['login'].str.strip()
saps_long = saps_long.drop_duplicates(subset=['idx', 'login'])
saps_filtrado = saps_long[saps_long['login'].isin(logins)].reset_index(drop=True)

prod_saps = pd.crosstab(
    index=saps_filtrado['login'],
    columns=saps_filtrado['MES'],
    margins=True, margins_name='TOTAL'
)
prod_saps = pd.concat([
    prod_saps.drop('TOTAL').sort_values(by='TOTAL', ascending=False),
    prod_saps.loc[['TOTAL']]
])
prod_saps = prod_saps.reset_index().rename(columns={'login': 'Usuario Mov.'})
prod_saps = pd.merge(prod_saps, info_pessoal, left_on='Usuario Mov.', right_on='LOGIN', how='left')
prod_saps = prod_saps.drop('LOGIN', axis=1)
colunas_mes_saps = [c for c in prod_saps.columns
                    if c not in ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor', 'TOTAL']]
prod_saps = prod_saps[['Usuario Mov.', 'Status', 'Tipo', 'Supervisor'] + colunas_mes_saps + ['TOTAL']]

print(f'SAPS filtrado: {len(prod_saps):,} operadores x {len(colunas_mes_saps)} meses')


SAPS filtrado: 331 operadores x 5 meses


In [7]:
# =============================================================
# PAINEL UNIFICADO: WORKFLOW + SAPS por equipe
# =============================================================

# Passo 1: Empilha as duas tabelas (sem linhas de TOTAL)
base_bruta_unificada = pd.concat([
    produtividade_filtrada[produtividade_filtrada['Usuario Mov.'] != 'TOTAL'],
    prod_saps[prod_saps['Usuario Mov.'] != 'TOTAL']
], ignore_index=True)

# Passo 2: Meses unificados e ordenados
colunas_mes_unificadas = sorted(list(set(colunas_mes + colunas_mes_saps)))
colunas_numericas_unificadas = colunas_mes_unificadas + ['TOTAL']

base_bruta_unificada[colunas_numericas_unificadas] = (
    base_bruta_unificada[colunas_numericas_unificadas]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

# Passo 3: Agrupa por funcionario somando Workflow + SAPS
prod_unificada = base_bruta_unificada.groupby(
    ['Usuario Mov.', 'Status', 'Tipo', 'Supervisor'], as_index=False
)[colunas_numericas_unificadas].sum()
prod_unificada[colunas_numericas_unificadas] = prod_unificada[colunas_numericas_unificadas].astype(int)

# Passo 4: Exibe por equipe (supervisor)
supervisores_unicos = prod_unificada['Supervisor'].dropna().unique()

for supervisor in supervisores_unicos:
    equipe = prod_unificada[prod_unificada['Supervisor'] == supervisor].copy()
    equipe = equipe.sort_values(by='TOTAL', ascending=False)

    total_equipe = equipe[colunas_numericas_unificadas].sum().astype(int)
    total_equipe['Usuario Mov.'] = 'TOTAL EQUIPE'
    total_equipe['Status'] = ''
    total_equipe['Tipo'] = ''
    total_equipe['Supervisor'] = ''

    equipe = pd.concat([equipe, pd.DataFrame([total_equipe])], ignore_index=True)
    equipe[colunas_numericas_unificadas] = equipe[colunas_numericas_unificadas].astype(int)

    print(f'\n====== EQUIPE UNIFICADA (WORKFLOW + SAPS): {supervisor} ======')
    display(equipe.style.hide(axis='index').format({col: '{:d}' for col in colunas_numericas_unificadas}))



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): LEONARDO RIBEIRO ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
ANADIAS,ATIVO,EFETIVO,LEONARDO RIBEIRO,0,30,384,264,351,226,66,1321
EVELYNG,ATIVO,EFETIVO,LEONARDO RIBEIRO,20,27,263,194,220,213,48,985
ROSANEN,ATIVO,EFETIVO,LEONARDO RIBEIRO,22,31,232,165,275,204,53,982
HERICA,ATIVO,EFETIVO,LEONARDO RIBEIRO,30,58,83,121,206,175,94,767
DANUZAA,ATIVO,EFETIVO,LEONARDO RIBEIRO,3,5,187,158,141,194,73,761
FAFONSO,ATIVO,EFETIVO,LEONARDO RIBEIRO,24,55,264,27,160,175,42,747
ADRIANAF,ATIVO,EFETIVO,LEONARDO RIBEIRO,34,50,171,110,105,145,65,680
DENISER,ATIVO,EFETIVO,LEONARDO RIBEIRO,28,59,160,80,139,111,57,634
MARTINSS,ATIVO,EFETIVO,LEONARDO RIBEIRO,20,53,175,80,148,129,20,625
AMARINHO,ATIVO,EFETIVO,LEONARDO RIBEIRO,13,18,146,143,81,147,50,598



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ANA PAULA MATTOS ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
JCRIST,ATIVO,EFETIVO,ANA PAULA MATTOS,20,41,490,453,447,491,128,2070
CRISMAR,ATIVO,EFETIVO,ANA PAULA MATTOS,0,24,318,284,355,306,100,1387
JOSIANEL,FERIAS,EFETIVO,ANA PAULA MATTOS,17,16,303,313,271,352,28,1300
ADRIANAO,FERIAS,EFETIVO,ANA PAULA MATTOS,23,37,285,234,365,300,28,1272
GLAUCI,ATIVO,EFETIVO,ANA PAULA MATTOS,27,63,303,211,293,273,68,1238
DAVIDM,ATIVO,EFETIVO,ANA PAULA MATTOS,14,24,336,264,335,196,46,1215
FVENTURA,FERIAS,EFETIVO,ANA PAULA MATTOS,18,41,315,309,348,160,0,1191
WAGNER,ATIVO,EFETIVO,ANA PAULA MATTOS,0,22,268,233,325,260,48,1156
ALEPP,ATIVO,EFETIVO,ANA PAULA MATTOS,35,2,291,191,292,218,84,1113
LBARA,FERIAS,EFETIVO,ANA PAULA MATTOS,15,38,227,243,296,245,37,1101



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ALAN VIEIRA ======


...

Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
ECARDOSO,ATIVO,EFETIVO,LAIS COSTA,2,1,6,5,4,0,0,18
ROBERT,ATIVO,EFETIVO,LAIS COSTA,0,0,0,0,1,0,0,1
TOTAL EQUIPE,,,,2,1,6,5,5,0,0,19



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ROSANGELA CAETANO ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
GRASSO,ATIVO,EFETIVO,ROSANGELA CAETANO,6,0,81,57,56,42,7,249
GQUEIROZ,ATIVO,EFETIVO,ROSANGELA CAETANO,9,10,0,44,70,52,32,217
TOTAL EQUIPE,,,,15,10,81,101,126,94,39,466



====== EQUIPE UNIFICADA (WORKFLOW + SAPS): ADRIANA NOBRE ======


Usuario Mov.,Status,Tipo,Supervisor,25-11,25-12,26-01,26-02,26-03,26-04,26-05,TOTAL
SUZANE,EMPRESTADO,EFETIVO,ADRIANA NOBRE,185,129,268,432,326,70,0,1410
TOTAL EQUIPE,,,,185,129,268,432,326,70,0,1410
